# Credit Card Fraud Detection

### Libraries and Dataset Import
We import the required libraries and load the training dataset. A quick look at the first rows gives an overview of the available features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler, QuantileTransformer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, roc_auc_score, average_precision_score,
                              ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
                              precision_recall_curve, f1_score)
import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset


In [ ]:
df = pd.read_csv("archive/fraudTrain.csv", index_col="Unnamed: 0")
df.head()

## Exploratory Data Analysis

We begin by familiarizing ourselves with the dataset: inspecting its shape and checking for missing or duplicate values. We also use `info()` to verify the data type of each column.

In [ ]:
print(f"Dataset size: {df.shape}")
print(f"Missing values: {df.isna().sum().sum()}")
print(f"Duplicate values: {df.duplicated().sum()}")

In [ ]:
df.info()

The dataset contains no missing or duplicate values. Among the 22 features, all data types are correctly inferred except for the two temporal columns `trans_date_trans_time` and `dob`, which must be converted to datetime before any further processing.

In [ ]:
df["trans_date_trans_time"]=pd.to_datetime(df["trans_date_trans_time"])
df["dob"]=pd.to_datetime(df["dob"])

We begin the EDA by examining the target variable `is_fraud` to assess the distribution of fraudulent transactions.

In [ ]:
frauds=df["is_fraud"].value_counts()
fig,ax=plt.subplots()
ax.bar(frauds.index,frauds.values)
ax.set_xticks(frauds.index)
ax.set_title("Frauds distribution")
ax.set_xlabel(f"0: Not Fraud ({round(frauds[0]/frauds.sum()*100,2)}%), 1: Fraud ({round(frauds[1]/frauds.sum()*100,2)}%)")
ax.set_ylabel("Occurrences")

plt.tight_layout()
plt.show()

As expected, fraudulent transactions represent a negligible fraction of the dataset (~0.58%). This severe class imbalance means we are not dealing with a standard binary classification problem, but rather an anomaly detection task. We proceed by inspecting `trans_date_trans_time` to identify any temporal patterns in fraudulent activity.

In [ ]:
trans_hour=df["trans_date_trans_time"].dt.hour
fraud_hour=df["trans_date_trans_time"][df["is_fraud"]==1].dt.hour
trans_hour_counts=trans_hour.value_counts().sort_index()
fraud_hour_counts=fraud_hour.value_counts().sort_index()
perc_frauds=fraud_hour_counts/trans_hour_counts*100

fig,ax=plt.subplots(ncols=2, figsize=(10,5))
ax[0].bar(np.arange(24),fraud_hour_counts)
ax[0].set_xticks(np.arange(0,24,2))
ax[0].set_xlabel("Hour of the day")
ax[0].set_ylabel("Number of frauds")
ax[0].set_title("Fraud occurrences per hour")

ax[1].bar(np.arange(24),perc_frauds)
ax[1].set_xticks(np.arange(0,24,2))
ax[1].set_xlabel("Hour of the day")
ax[1].set_ylabel("% of frauds")
ax[1].set_title("% of frauds per hour")

plt.tight_layout()
plt.show()


The first important signal emerges clearly: fraud is strongly concentrated between 10 PM and 3 AM. This holds both in absolute count and as a percentage of total transactions per hour, ruling out any confound from varying transaction volumes throughout the day.

We further decompose the temporal dimension to check whether the day of the week, the day of the month, and the month of the year carry additional signal.

In [ ]:
fig, ax = plt.subplots(nrows=3, ncols=2,figsize=(10,8))

trans_dow = df["trans_date_trans_time"].dt.dayofweek
fraud_dow = df["trans_date_trans_time"][df["is_fraud"]==1].dt.dayofweek
trans_dow_counts = trans_dow.value_counts().sort_index()
fraud_dow_counts = fraud_dow.value_counts().sort_index()
perc_frauds_dow = (fraud_dow_counts / trans_dow_counts) * 100

dow = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

ax[0, 0].bar(trans_dow_counts.index, fraud_dow_counts)
ax[0, 0].set_xticks(range(7))
ax[0, 0].set_xticklabels(dow)
ax[0, 0].set_ylabel("Number of frauds")
ax[0, 0].set_title("Fraud occurrences per day of week")

ax[0, 1].bar(trans_dow_counts.index, perc_frauds_dow)
ax[0, 1].set_xticks(range(7))
ax[0, 1].set_xticklabels(dow)
ax[0, 1].set_ylabel("% of frauds")
ax[0, 1].set_title("% of frauds per day of week")

trans_dom = df["trans_date_trans_time"].dt.day
fraud_dom = df["trans_date_trans_time"][df["is_fraud"]==1].dt.day
trans_dom_counts = trans_dom.value_counts().sort_index()
fraud_dom_counts = fraud_dom.value_counts().sort_index()
perc_frauds_dom = (fraud_dom_counts / trans_dom_counts) * 100

ax[1, 0].bar(trans_dom_counts.index, fraud_dom_counts)
ax[1, 0].set_xticks(range(1, 32, 2))
ax[1, 0].set_xlabel("Day of month")
ax[1, 0].set_ylabel("Number of frauds")
ax[1, 0].set_title("Fraud occurrences per day of month")

ax[1, 1].bar(trans_dom_counts.index, perc_frauds_dom)
ax[1, 1].set_xticks(range(1, 32, 2))
ax[1, 1].set_xlabel("Day of month")
ax[1, 1].set_ylabel("% of frauds")
ax[1, 1].set_title("% of frauds per day of month")

trans_month = df["trans_date_trans_time"].dt.month
fraud_month = df["trans_date_trans_time"][df["is_fraud"] == 1].dt.month
trans_month_counts = trans_month.value_counts().sort_index()
fraud_month_counts = fraud_month.value_counts().sort_index()
perc_frauds_month = (fraud_month_counts / trans_month_counts) * 100

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']


ax[2,0].bar(fraud_month_counts.index, fraud_month_counts.values)
ax[2,0].set_xticks(range(1, 13))
ax[2,0].set_xticklabels(months)
ax[2,0].set_xlabel("Month")
ax[2,0].set_ylabel("Number of frauds")
ax[2,0].set_title("Fraud occurrences per month")

ax[2,1].bar(perc_frauds_month.index, perc_frauds_month.values)
ax[2,1].set_xticks(range(1, 13))
ax[2,1].set_xticklabels(months)
ax[2,1].set_xlabel("Month")
ax[2,1].set_ylabel("% of frauds")
ax[2,1].set_title("% of frauds per month")

plt.tight_layout()
plt.show()

Broadening the temporal granularity yields no additional signal. Neither the day of the week, the day of the month, nor the month of the year show a consistent pattern in the fraud rate. The hour of the day remains the only informative temporal feature and will be extracted during feature engineering.

In [ ]:
trans_cat = df["category"].value_counts()
fraud_cat = df[df["is_fraud"] == 1]["category"].value_counts()
perc_frauds_cat = (fraud_cat / trans_cat * 100).dropna().sort_values(ascending=True)
fraud_cat_sorted = fraud_cat.sort_values(ascending=True)

fig, ax = plt.subplots(ncols=2, figsize=(15, 7))

ax[0].barh(fraud_cat_sorted.index, fraud_cat_sorted.values)
ax[0].set_xlabel("Number of frauds")
ax[0].set_title("Top 10 categories by number of frauds")

ax[1].barh(perc_frauds_cat.index, perc_frauds_cat.values)
ax[1].set_xlabel("% of frauds")
ax[1].set_title("Top 10 categories by % of frauds")

plt.tight_layout()
plt.show()

A clear signal emerges: while the most represented categories in absolute terms are everyday ones such as groceries and shopping, the fraud rate is dominated by online transactions — those belonging to categories ending with the *_net* suffix.

Before moving to the next feature, we briefly inspect `merchant` to verify whether it carries any signal beyond what `category` already captures.

In [ ]:
trans_merch = df['merchant'].value_counts()
fraud_merch = df[df['is_fraud'] == 1]['merchant'].value_counts()

top20_abs = fraud_merch.sort_values(ascending=True).tail(10)
top20_perc = (fraud_merch / trans_merch * 100).dropna().sort_values(ascending=True).tail(10)

fig, ax = plt.subplots(ncols=2, figsize=(15, 7))

ax[0].barh(top20_abs.index, top20_abs.values)
ax[0].set_xlabel("Number of frauds")
ax[0].set_title("Top 10 merchants by number of frauds")

ax[1].barh(top20_perc.index, top20_perc.values)
ax[1].set_xlabel("% of frauds")
ax[1].set_title("Top 10 merchants by % of frauds")

plt.tight_layout()
plt.show()

The `merchant` feature has approximately 700 unique values. We limit the analysis to the top 10 merchants by absolute fraud count and by fraud rate, following the same dual-perspective approach used throughout the EDA.

No consistent signal emerges from the merchant-level analysis. The merchants with the highest fraud rates are those with very few total transactions — a statistical artefact rather than a genuine pattern. With ~700 unique values and no signal beyond what `category` already captures at cardinality 14, `merchant` will be dropped.

We now turn to the `amt` feature, which represents the transaction amount in dollars.

In [ ]:
print("Stats legit transactions")
print(df[df['is_fraud'] == 0]['amt'].describe())
print("\nStats frauds")
print(df[df['is_fraud'] == 1]['amt'].describe())

fig, ax = plt.subplots(ncols=2, figsize=(15, 6))

ax[0].boxplot([df[df["is_fraud"]==0]["amt"], df[df["is_fraud"]==1]["amt"]], tick_labels=["Not Fraud", "Fraud"])
ax[0].set_yscale("log") 
ax[0].set_ylabel("Amount ($, log scale)")
ax[0].set_title("Transactions amounts boxplot (log scale)")

ax[1].hist(df[(df["is_fraud"]==0) & (df["amt"] < 1400)]["amt"], bins=50, alpha=0.5, label="Not Fraud", density=True)
ax[1].hist(df[(df["is_fraud"]==1) & (df["amt"] < 1400)]["amt"], bins=50, alpha=0.5, label="Fraud", density=True)
ax[1].set_xlabel("Amount ($)")
ax[1].set_ylabel("Frauds density")
ax[1].set_title("Amounts distributions (Zoom < 1400$)")
ax[1].legend()

plt.tight_layout()
plt.show()

The printed statistics, boxplots, and empirical distributions converge on a clear picture: legitimate transactions are concentrated in the range of tens of dollars, with a long but thin tail of occasional large purchases. Fraudulent transactions show a distinctly different profile — fraudsters typically begin with small test charges to verify the card is active, then follow up with larger transactions in the hundreds of dollars.

We next examine `dob` to check whether the cardholder's age is associated with fraud likelihood.

In [ ]:
trans_yob = df["dob"].dt.year
fraud_yob = df[df["is_fraud"] == 1]["dob"].dt.year
trans_yob_counts = trans_yob.value_counts().sort_index()
fraud_yob_counts = fraud_yob.value_counts().sort_index()
perc_frauds_yob = (fraud_yob_counts / trans_yob_counts) * 100

fig, ax = plt.subplots(ncols=2, figsize=(15, 5))

ax[0].bar(fraud_yob_counts.index, fraud_yob_counts.values)
ax[0].set_xlabel("Year of birth")
ax[0].set_ylabel("Number of frauds")
ax[0].set_title("Number of frauds per year of birth")


ax[1].bar(perc_frauds_yob.index, perc_frauds_yob.values)
ax[1].set_xlabel("Year of birth")
ax[1].set_ylabel("% of frauds")
ax[1].set_yscale("log")
ax[1].set_title("% of frauds per year of birth")


plt.tight_layout()
plt.show()

In [ ]:
df[df["dob"].dt.year==1925][["cc_num","is_fraud"]].value_counts()

The absolute fraud count by birth year simply reflects the active population (roughly those born between 1960 and 2000), with no age-related pattern in the fraud rate. The spike at 1925 appears anomalous but is entirely explained by a single compromised card — confirmed by the lookup above. Age carries no predictive signal and `dob` will be dropped.

We now check whether the distance between the cardholder's home location (`lat`, `long`) and the merchant's location (`merch_lat`, `merch_long`) is informative.

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0 
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    d=R*c
    return d

dist_km = haversine_distance(df['lat'], df['long'], df['merch_lat'], df['merch_long'])

fig, ax = plt.subplots(ncols=2, figsize=(15, 6))

ax[0].boxplot([dist_km[df["is_fraud"]==0], dist_km[df["is_fraud"]==1]], tick_labels=["Not Fraud", "Fraud"])
ax[0].set_ylabel("Distance (km)")
ax[0].set_title("Distance user-commerce")

ax[1].hist(dist_km[df["is_fraud"]==0], bins=50, alpha=0.5, label="Not Fraud", density=True)
ax[1].hist(dist_km[df["is_fraud"]==1], bins=50, alpha=0.5, label="Fraud", density=True)
ax[1].set_xlabel("Distance (km)")
ax[1].set_ylabel("Frauds density")
ax[1].set_title("Distances distributions")
ax[1].legend()

plt.tight_layout()
plt.show()

We compute the great-circle distance between the cardholder's registered address and the merchant location using the Haversine formula. Both the boxplots and the empirical distributions are nearly identical for fraudulent and legitimate transactions, confirming that geographical distance carries no useful signal.

We now check whether the cardholder's `gender` is a relevant predictor.

In [ ]:
trans_gender = df['gender'].value_counts().sort_index()
fraud_gender = df[df['is_fraud']==1]['gender'].value_counts().sort_index()
perc_gender = (fraud_gender / trans_gender) * 100

fig, ax = plt.subplots(ncols=2, figsize=(14, 5))

ax[0].bar(fraud_gender.index, fraud_gender.values)
ax[0].set_xlabel("Gender")
ax[0].set_ylabel("Number of frauds")
ax[0].set_title("Number of frauds per gender")

ax[1].bar(perc_gender.index, perc_gender.values)
ax[1].set_xlabel("Gender")
ax[1].set_ylabel("% of frauds")
ax[1].set_title("% of frauds per gender")

plt.tight_layout()
plt.show()

Gender carries no predictive signal: the fraud rate is essentially identical across both groups.

We next examine `city_pop`, the population of the cardholder's city of residence.

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(14, 5))

ax[0].boxplot([df[df["is_fraud"]==0]["city_pop"], df[df["is_fraud"]==1]["city_pop"]], tick_labels=["Not Fraud", "Fraud"])
ax[0].set_yscale("log")
ax[0].set_ylabel("Population (Log Scale)")
ax[0].set_title("Population boxplots (Log Scale)")

bins = np.logspace(np.log10(df['city_pop'].min()), np.log10(df['city_pop'].max()), 50)
ax[1].hist(df[df["is_fraud"]==0]["city_pop"], bins=bins, alpha=0.5, label="Not Fraud", density=True)
ax[1].hist(df[df["is_fraud"]==1]["city_pop"], bins=bins, alpha=0.5, label="Fraud", density=True)
ax[1].set_xscale("log")
ax[1].set_xlabel("Population (Log Scale)")
ax[1].set_ylabel("Population density")
ax[1].set_title("Population densities distribtions (Log Scale)")
ax[1].legend()

plt.tight_layout()
plt.show()

The distributions of city population are nearly identical for fraudulent and legitimate transactions — no useful signal here either.

To complete the geographical analysis, we examine fraud rates by US state. We deliberately avoid analysing raw coordinates at finer granularity, as the high dimensionality would introduce noise rather than signal.

In [ ]:
trans_state = df['state'].value_counts()
fraud_state = df[df['is_fraud']==1]['state'].value_counts()

top10_abs = fraud_state.sort_values(ascending=False).head(10)
perc_state = (fraud_state / trans_state * 100).sort_values(ascending=False).head(10)

fig, ax = plt.subplots(ncols=2, figsize=(14, 5))

ax[0].bar(top10_abs.index, top10_abs.values)
ax[0].set_xlabel("State")
ax[0].set_ylabel("Number of frauds")
ax[0].set_title("Top 10 state per number of frauds")

ax[1].bar(perc_state.index, perc_state.values)
ax[1].set_xlabel("State")
ax[1].set_ylabel("% of frauds")
ax[1].set_title("Top 10 states per % of frauds")

plt.tight_layout()
plt.show()

In [ ]:
df[df['state']=='DE'][["cc_num","state","is_fraud"]].value_counts()

In absolute terms, no state shows a disproportionately high fraud count. Delaware (DE) appears to stand out in relative terms, but a closer look reveals that a single compromised card accounts for all fraudulent transactions in that state — the same statistical artefact encountered with the 1925 birth year. No geographical feature at any granularity carries actionable signal.

We conclude the EDA by analysing transaction velocity: the time elapsed between consecutive transactions on the same card.

In [ ]:
df_vel = df[['cc_num', 'unix_time', 'is_fraud']].sort_values(['cc_num', 'unix_time'])
df_vel['hours_since_last'] = df_vel.groupby('cc_num')['unix_time'].diff() / 3600

legit_intervals = df_vel[df_vel['is_fraud'] == 0]['hours_since_last'].dropna()
fraud_intervals = df_vel[df_vel['is_fraud'] == 1]['hours_since_last'].dropna()

clip_h = 100
fig, ax = plt.subplots(ncols=2, figsize=(15, 5))

ax[0].boxplot([legit_intervals, fraud_intervals], tick_labels=['Not Fraud', 'Fraud'])
ax[0].set_yscale('log')
ax[0].set_ylabel('Hours (log scale)')
ax[0].set_title('Time between consecutive transactions per card (log scale)')

ax[1].hist(legit_intervals.clip(upper=clip_h), bins=50, alpha=0.5, label='Not Fraud', density=True)
ax[1].hist(fraud_intervals.clip(upper=clip_h), bins=50, alpha=0.5, label='Fraud', density=True)
ax[1].set_xlabel(f'Hours since previous transaction on same card (clipped at {clip_h}h)')
ax[1].set_ylabel('Density')
ax[1].set_title('Transaction velocity per card')
ax[1].legend()

plt.tight_layout()
plt.show()

print(f"Median hours since last transaction — Not Fraud: {legit_intervals.median():.1f}h")
print(f"Median hours since last transaction — Fraud:     {fraud_intervals.median():.1f}h")

Fraudulent transactions occur significantly closer together in time than legitimate ones: the median interval drops from 4.6 hours for legitimate transactions to 1.4 hours for fraudulent ones. This is consistent with the typical fraud pattern — a burst of rapid charges once a card is stolen. The time elapsed since the last transaction on the same card (`hours_since_last_trans`) is a strong candidate for feature engineering.

The raw `amt` feature carries meaningful signal, but its discriminative power increases when conditioned on the purchase category. A transaction of $500 is unremarkable for `shopping_net` but anomalous for `gas_transport`. We engineer `amt_zscore_per_category` as the z-score of `amt` within each category, computed using the mean and standard deviation of legitimate transactions only — making it a measure of how unusual a given amount is relative to the typical spend in that category.

In [ ]:
cat_mean = df[df['is_fraud'] == 0].groupby('category')['amt'].mean()
cat_std  = df[df['is_fraud'] == 0].groupby('category')['amt'].std()

amt_z   = (df['amt'] - df['category'].map(cat_mean)) / (df['category'].map(cat_std) + 1e-6)
legit_z = amt_z[df['is_fraud'] == 0]
fraud_z = amt_z[df['is_fraud'] == 1]

clip_z = 10
fig, ax = plt.subplots(ncols=2, figsize=(14, 5))

ax[0].boxplot([legit_z.clip(-clip_z, clip_z), fraud_z.clip(-clip_z, clip_z)],
              tick_labels=['Not Fraud', 'Fraud'])
ax[0].set_ylabel('Amount z-score per category')
ax[0].set_title('amt_zscore_per_category')

ax[1].hist(legit_z.clip(-clip_z, clip_z), bins=60, alpha=0.5, label='Not Fraud', density=True)
ax[1].hist(fraud_z.clip(-clip_z, clip_z), bins=60, alpha=0.5, label='Fraud', density=True)
ax[1].set_xlabel('Z-score (clipped at ±10)')
ax[1].set_ylabel('Density')
ax[1].set_title('Distribution of amount z-score per category')
ax[1].legend()

plt.tight_layout()
plt.show()

print(f"Median   — Not Fraud: {legit_z.median():.2f}   Fraud: {fraud_z.median():.2f}")


`amt_zscore_per_category` normalises the amount relative to the category — but two cardholders in the same category can have very different spending habits. A $300 charge on `misc_net` may be typical for one cardholder and three standard deviations above their norm for another. We engineer `amt_zscore_per_card` as the z-score of `amt` relative to each cardholder's own historical mean and standard deviation, offering a complementary and orthogonal signal: how anomalous is this amount *for this specific card*.

In [ ]:
card_mean  = df.groupby('cc_num')['amt'].transform('mean')
card_std   = df.groupby('cc_num')['amt'].transform('std')
amt_z_card = (df['amt'] - card_mean) / (card_std + 1e-6)

legit_zc = amt_z_card[df['is_fraud'] == 0]
fraud_zc = amt_z_card[df['is_fraud'] == 1]

clip_z = 10
fig, ax = plt.subplots(ncols=2, figsize=(14, 5))

ax[0].boxplot([legit_zc.clip(-clip_z, clip_z), fraud_zc.clip(-clip_z, clip_z)],
              tick_labels=['Not Fraud', 'Fraud'])
ax[0].set_ylabel('Amount z-score per card')
ax[0].set_title('amt_zscore_per_card')

ax[1].hist(legit_zc.clip(-clip_z, clip_z), bins=60, alpha=0.5, label='Not Fraud', density=True)
ax[1].hist(fraud_zc.clip(-clip_z, clip_z), bins=60, alpha=0.5, label='Fraud', density=True)
ax[1].set_xlabel('Z-score (clipped at ±10)')
ax[1].set_ylabel('Density')
ax[1].set_title('Distribution of amount z-score per card')
ax[1].legend()

plt.tight_layout()
plt.show()

print(f"Median   — Not Fraud: {legit_zc.median():.2f}   Fraud: {fraud_zc.median():.2f}")

## Metrics evaluation function

In [ ]:
def evaluate_model(y_val, y_proba, model_name="Model", threshold=0.5):
    y_pred  = (y_proba >= threshold).astype(int)

    print(f"=== {model_name} ===")
    print(classification_report(y_val, y_pred, target_names=['Not Fraud', 'Fraud']))
    print(f"ROC-AUC : {roc_auc_score(y_val, y_proba):.4f}")
    print(f"PR-AUC  : {average_precision_score(y_val, y_proba):.4f}")

    fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(20, 12))

    ConfusionMatrixDisplay.from_predictions(y_val, y_pred, ax=ax[0, 0], display_labels=['Not Fraud', 'Fraud'])
    ax[0, 0].set_title(f'{model_name} — Confusion Matrix')

    RocCurveDisplay.from_predictions(y_val, y_proba, ax=ax[0, 1])
    ax[0, 1].set_title(f'{model_name} — ROC Curve')

    PrecisionRecallDisplay.from_predictions(y_val, y_proba, ax=ax[0, 2])
    ax[0, 2].set_title(f'{model_name} — Precision-Recall Curve')

    ax[1, 0].hist(y_proba[np.array(y_val) == 0], bins=100, alpha=0.5, label='Not Fraud', density=True)
    ax[1, 0].hist(y_proba[np.array(y_val) == 1], bins=100, alpha=0.5, label='Fraud', density=True)
    ax[1, 0].set_yscale('log')
    ax[1, 0].set_xlabel('Anomaly score')
    ax[1, 0].set_title(f'{model_name} — Score Distribution')
    ax[1, 0].legend()

    precision, recall, thresholds = precision_recall_curve(y_val, y_proba)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    ax[1, 1].plot(thresholds, precision[:-1], label='Precision')
    ax[1, 1].plot(thresholds, recall[:-1], label='Recall')
    ax[1, 1].plot(thresholds, f1[:-1], label='F1')
    ax[1, 1].set_xlabel('Threshold')
    ax[1, 1].set_title(f'{model_name} — Metrics vs Threshold')
    ax[1, 1].legend()

    ax[1, 2].axis('off')

    plt.tight_layout()
    plt.show()

## Supervised Baseline — Logistic Regression

Logistic Regression establishes the linear supervised ceiling. With `class_weight='balanced'` the model reweights the minority class to compensate for the ~170:1 imbalance. Log-transforming `amt` and `hours_since_last_trans` linearises their right-skewed distributions, making them accessible to a linear decision boundary. Any non-linear method should comfortably surpass this baseline.

### Feature Engineering

In [ ]:
df_baseline = df.copy()
df_baseline = df_baseline.sort_values(['cc_num', 'unix_time'])

df_baseline['hours_since_last_trans'] = df_baseline.groupby('cc_num')['unix_time'].diff() / 3600
df_baseline['hours_since_last_trans'] = df_baseline['hours_since_last_trans'].fillna(
    df_baseline['hours_since_last_trans'].median()
)

_hour = df_baseline['trans_date_trans_time'].dt.hour
df_baseline['hour_sin'] = np.sin(2 * np.pi * _hour / 24)
df_baseline['hour_cos'] = np.cos(2 * np.pi * _hour / 24)

_cat_mean = df[df['is_fraud'] == 0].groupby('category', observed=True)['amt'].mean()
_cat_std  = df[df['is_fraud'] == 0].groupby('category', observed=True)['amt'].std()
# amount deviation relative to the category mean — captures unusual spend for the purchase type
df_baseline['amt_zscore_cat'] = (
    (df_baseline['amt'] - df_baseline['category'].map(_cat_mean))
    / (df_baseline['category'].map(_cat_std) + 1e-6)
)

# amount deviation relative to this card's own history — captures spend anomaly per cardholder
_g = df_baseline.groupby('cc_num')['amt']
df_baseline['amt_zscore_card'] = (
    (df_baseline['amt'] - _g.transform(lambda x: x.expanding().mean().shift(1)))
    / (_g.transform(lambda x: x.expanding().std().shift(1)) + 1e-6)
).fillna(0)

df_baseline['amt'] = np.log1p(df_baseline['amt'])
df_baseline['hours_since_last_trans'] = np.log1p(df_baseline['hours_since_last_trans'])

cols_to_drop = ['trans_num', 'cc_num', 'first', 'last', 'gender',
                'street', 'city', 'state', 'zip', 'lat', 'long',
                'merch_lat', 'merch_long', 'city_pop', 'job',
                'merchant', 'unix_time', 'dob', 'trans_date_trans_time']
df_baseline = df_baseline.drop(columns=cols_to_drop).sort_index()


### Preprocessing

In [ ]:
X = df_baseline.drop(columns=['is_fraud'])
y = df_baseline['is_fraud']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=False)

num_features = ['amt', 'hours_since_last_trans', 'hour_sin', 'hour_cos', 'amt_zscore_cat', 'amt_zscore_card']
cat_features = ['category']

preprocessor = ColumnTransformer([
    ('ohe', OneHotEncoder(handle_unknown='ignore'), cat_features),
    ('scaler', StandardScaler(), num_features)
])

pipe = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1))
])

pipe.fit(X_train, y_train)


### LogReg - Model evalutaion

In [ ]:
y_proba_lr = pipe.predict_proba(X_val)[:, 1]
evaluate_model(y_val, y_proba_lr, "Logistic Regression Baseline")

## Supervised Advanced — LightGBM

LightGBM is a gradient-boosted tree framework optimised for large tabular datasets. It captures non-linear feature interactions natively without requiring manual transformations — `hour` is kept raw rather than cyclically encoded since tree splits handle the 23→0 boundary automatically. The `scale_pos_weight` parameter rebalances the positive class in proportion to the training class ratio. Early stopping on the validation set prevents overfitting beyond the optimal tree count.

### Feature Engineering

In [ ]:
df_lgbm = df.copy()
df_lgbm = df_lgbm.sort_values(['cc_num', 'unix_time'])

df_lgbm['hours_since_last_trans'] = df_lgbm.groupby('cc_num')['unix_time'].diff() / 3600
df_lgbm['hours_since_last_trans'] = df_lgbm['hours_since_last_trans'].fillna(
    df_lgbm['hours_since_last_trans'].median()
)

df_lgbm['hour'] = df_lgbm['trans_date_trans_time'].dt.hour

_cat_mean = df[df['is_fraud'] == 0].groupby('category', observed=True)['amt'].mean()
_cat_std  = df[df['is_fraud'] == 0].groupby('category', observed=True)['amt'].std()
# amount deviation relative to the category mean — captures unusual spend for the purchase type
df_lgbm['amt_zscore_cat'] = (
    (df_lgbm['amt'] - df_lgbm['category'].map(_cat_mean))
    / (df_lgbm['category'].map(_cat_std) + 1e-6)
)

# amount deviation relative to this card's own history — captures spend anomaly per cardholder
_g = df_lgbm.groupby('cc_num')['amt']
df_lgbm['amt_zscore_card'] = (
    (df_lgbm['amt'] - _g.transform(lambda x: x.expanding().mean().shift(1)))
    / (_g.transform(lambda x: x.expanding().std().shift(1)) + 1e-6)
).fillna(0)

cols_to_drop = ['trans_num', 'cc_num', 'first', 'last', 'gender',
                'street', 'city', 'state', 'zip', 'lat', 'long',
                'merch_lat', 'merch_long', 'city_pop', 'job',
                'merchant', 'unix_time', 'dob', 'trans_date_trans_time']
df_lgbm = df_lgbm.drop(columns=cols_to_drop)
df_lgbm['category'] = df_lgbm['category'].astype('category')  # cast after FE, required for LGBM native handling
df_lgbm = df_lgbm.sort_index()


### Preprocessing

In [ ]:
X_lgbm = df_lgbm.drop(columns=['is_fraud'])
y_lgbm = df_lgbm['is_fraud']

X_train_lgbm, X_val_lgbm, y_train_lgbm, y_val_lgbm = train_test_split(X_lgbm, y_lgbm, test_size=0.2, shuffle=False)

scale_pos = (y_train_lgbm == 0).sum() / (y_train_lgbm == 1).sum()

lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    scale_pos_weight=scale_pos,
    random_state=42,
    n_jobs=-1
)

lgbm.fit(
    X_train_lgbm, y_train_lgbm,
    categorical_feature=['category'],
    eval_set=[(X_val_lgbm, y_val_lgbm)],
    callbacks=[early_stopping(50, verbose=False), log_evaluation(100)]
)


### LightGBM - Model evalutation 

In [ ]:
y_proba_lgbm = lgbm.predict_proba(X_val_lgbm)[:, 1]
evaluate_model(y_val_lgbm, y_proba_lgbm, "LightGBM")

## Unsupervised Models

Both unsupervised models are trained exclusively on normal transactions and assign anomaly scores without label supervision. The feature engineering and preprocessing steps below are shared by both models.

### Feature Engineering

In [ ]:
df_ae = df.copy()
df_ae = df_ae.sort_values(['cc_num', 'unix_time'])

df_ae['hours_since_last_trans'] = df_ae.groupby('cc_num')['unix_time'].diff() / 3600
df_ae['hours_since_last_trans'] = df_ae['hours_since_last_trans'].fillna(
    df_ae['hours_since_last_trans'].median()
)

_hour = df_ae['trans_date_trans_time'].dt.hour
df_ae['hour_sin'] = np.sin(2 * np.pi * _hour / 24)
df_ae['hour_cos'] = np.cos(2 * np.pi * _hour / 24)

_cat_mean = df[df['is_fraud'] == 0].groupby('category', observed=True)['amt'].mean()
_cat_std  = df[df['is_fraud'] == 0].groupby('category', observed=True)['amt'].std()
# amount deviation relative to the category mean — captures unusual spend for the purchase type
df_ae['amt_zscore_cat'] = (
    (df_ae['amt'] - df_ae['category'].map(_cat_mean))
    / (df_ae['category'].map(_cat_std) + 1e-6)
)

# amount deviation relative to this card's own history — captures spend anomaly per cardholder
_g = df_ae.groupby('cc_num')['amt']
df_ae['amt_zscore_card'] = (
    (df_ae['amt'] - _g.transform(lambda x: x.expanding().mean().shift(1)))
    / (_g.transform(lambda x: x.expanding().std().shift(1)) + 1e-6)
).fillna(0)

cols_to_drop = [
    'trans_num', 'cc_num', 'first', 'last', 'gender',
    'street', 'city', 'state', 'zip', 'lat', 'long',
    'merch_lat', 'merch_long', 'city_pop', 'job',
    'merchant', 'unix_time', 'dob', 'trans_date_trans_time'
]
df_ae = df_ae.drop(columns=cols_to_drop).sort_index()


### Preprocessing

In [ ]:
NUM_FEATURES = ['amt', 'hours_since_last_trans', 'hour_sin', 'hour_cos', 'amt_zscore_cat', 'amt_zscore_card']
CAT_FEATURES = ['category']
EMB_DIMS     = {'category': 4}

X_ae = df_ae.drop(columns=['is_fraud'])
y_ae = df_ae['is_fraud']

X_train_ae, X_val_ae, y_train_ae, y_val_ae = train_test_split(
    X_ae, y_ae, test_size=0.2, shuffle=False
)
X_train_normal = X_train_ae[y_train_ae == 0].copy()

qt_ae = QuantileTransformer(output_distribution='normal', n_quantiles=1000, random_state=42)
qt_ae.fit(X_train_normal[NUM_FEATURES])

cat_vocabs      = {}
cat_vocab_sizes = {}
for col in CAT_FEATURES:
    vals = sorted(X_train_normal[col].dropna().unique())
    cat_vocabs[col]      = {v: i + 1 for i, v in enumerate(vals)}
    cat_vocab_sizes[col] = len(vals) + 1

def encode_cats(df_sub, vocabs, cols):
    arr = np.zeros((len(df_sub), len(cols)), dtype=np.int64)
    for j, col in enumerate(cols):
        arr[:, j] = df_sub[col].map(vocabs[col]).fillna(0).astype(int).values
    return arr

X_train_num = qt_ae.transform(X_train_normal[NUM_FEATURES]).astype(np.float32)
X_val_num   = qt_ae.transform(X_val_ae[NUM_FEATURES]).astype(np.float32)
X_train_cat = encode_cats(X_train_normal, cat_vocabs, CAT_FEATURES)
X_val_cat   = encode_cats(X_val_ae,       cat_vocabs, CAT_FEATURES)


## Unsupervised Baseline — Isolation Forest

Isolation Forest partitions the feature space with random splits and scores each observation by the average path length needed to isolate it: anomalous points are statistically distant from the bulk and therefore isolated with fewer splits. Like the Autoencoder, it is trained on normal transactions only and operates without label supervision. The same preprocessing pipeline (QuantileTransformer + OHE for `category`) is shared with the Autoencoder to ensure a fair comparison.

In [ ]:
ohe_if        = OneHotEncoder(handle_unknown='ignore')
X_tr_cat_ohe  = ohe_if.fit_transform(X_train_normal[['category']]).toarray()
X_val_cat_ohe = ohe_if.transform(X_val_ae[['category']]).toarray()

X_tr_if  = np.hstack([X_train_num, X_tr_cat_ohe])
X_val_if = np.hstack([X_val_num,   X_val_cat_ohe])

iso_forest = IsolationForest(n_estimators=200, random_state=42, n_jobs=-1)
iso_forest.fit(X_tr_if)

scores_if_val   = -iso_forest.score_samples(X_val_if)
scores_if_train = -iso_forest.score_samples(X_tr_if)

for pct in [99.0, 99.5, 99.9]:
    t  = np.percentile(scores_if_train, pct)
    f1 = f1_score(y_val_ae.values, (scores_if_val >= t).astype(int), zero_division=0)
    print(f"  {pct:.1f}th pct  thr={t:.4f}  F1={f1:.4f}")

thr_if = np.percentile(scores_if_train, 99.5)
evaluate_model(y_val_ae.values, scores_if_val, "Isolation Forest", threshold=thr_if)


## Unsupervised Advanced — Mixed-Data Autoencoder

The Mixed-Data Autoencoder extends the Isolation Forest baseline with a neural architecture: continuous features are reconstructed via Gaussian NLL (μ + log σ heads), categorical features via cross-entropy over learned embeddings, with denoising regularisation. The per-sample mixed reconstruction cost serves as the anomaly score.

### Architecture

In [ ]:
class MixedAutoencoder(nn.Module):
    def __init__(self, vocab_sizes, emb_dims, cat_cols, n_num, latent_dim=16, hidden_dim=64):
        super().__init__()
        self.cat_cols = cat_cols
        self.n_num    = n_num

        self.embeddings = nn.ModuleList([
            nn.Embedding(vocab_sizes[col], emb_dims[col]) for col in cat_cols
        ])

        input_dim = n_num + sum(emb_dims[col] for col in cat_cols)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
        self.decoder_trunk = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
        )
        self.cont_head = nn.Linear(hidden_dim, n_num * 2)
        self.cat_heads = nn.ModuleList([
            nn.Linear(hidden_dim, vocab_sizes[col]) for col in cat_cols
        ])

    def forward(self, x_num, x_cat):
        embs = [self.embeddings[i](x_cat[:, i]) for i in range(len(self.cat_cols))]
        x    = torch.cat(embs + [x_num], dim=1)
        z    = self.encoder(x)
        h    = self.decoder_trunk(z)
        cont = self.cont_head(h)
        mu        = cont[:, :self.n_num]
        log_sigma = cont[:, self.n_num:].clamp(-5.0, 2.0)
        return mu, log_sigma, [head(h) for head in self.cat_heads]


def mixed_nll_loss(x_num, x_cat, mu, log_sigma, cat_logits, cont_weight):
    sigma    = log_sigma.exp()
    nll_cont = (log_sigma + 0.5 * ((x_num - mu) / sigma) ** 2).sum(dim=1).mean()
    nll_cat  = torch.stack([
        F.cross_entropy(cat_logits[j], x_cat[:, j], reduction='none')
        for j in range(x_cat.shape[1])
    ], dim=1).sum(dim=1).mean()
    return cont_weight * nll_cont + (1 - cont_weight) * nll_cat


def compute_anomaly_scores(model, x_num_t, x_cat_t, cont_weight, batch_size=4096):
    model.eval()
    scores = []
    with torch.no_grad():
        for i in range(0, len(x_num_t), batch_size):
            xn, xc = x_num_t[i:i+batch_size], x_cat_t[i:i+batch_size]
            mu, log_sigma, cat_logits = model(xn, xc)
            sigma  = log_sigma.exp()
            s_cont = (log_sigma + 0.5 * ((xn - mu) / sigma) ** 2).sum(dim=1)
            s_cat  = torch.stack([
                F.cross_entropy(cat_logits[j], xc[:, j], reduction='none')
                for j in range(xc.shape[1])
            ], dim=1).sum(dim=1)
            scores.append((cont_weight * s_cont + (1 - cont_weight) * s_cat).cpu())
    return torch.cat(scores).numpy()


def threshold_from_normal_scores(scores_normal, percentile=99.5):
    return float(np.percentile(scores_normal, percentile))


### Model Training
`cont_weight=0.75`, `num_noise=0.025`, `cat_noise=0.10`. Denoising applied to inputs; loss computed against clean targets.

In [ ]:
torch.manual_seed(42)

CONT_WEIGHT = 0.75
NUM_NOISE   = 0.025
CAT_NOISE   = 0.10
EPOCHS      = 30

X_tr_num_t  = torch.FloatTensor(X_train_num)
X_tr_cat_t  = torch.LongTensor(X_train_cat)
X_val_num_t = torch.FloatTensor(X_val_num)
X_val_cat_t = torch.LongTensor(X_val_cat)
y_val_np    = y_val_ae.values

loader = DataLoader(
    TensorDataset(X_tr_num_t, X_tr_cat_t),
    batch_size=256, shuffle=True
)

model_ae     = MixedAutoencoder(
    vocab_sizes=cat_vocab_sizes, emb_dims=EMB_DIMS,
    cat_cols=CAT_FEATURES,      n_num=len(NUM_FEATURES),
)
optimizer_ae = optim.Adam(model_ae.parameters(), lr=1e-3)

train_losses = []
model_ae.train()
for epoch in range(EPOCHS):
    epoch_loss = []
    for x_num_b, x_cat_b in loader:
        x_num_in = x_num_b + torch.randn_like(x_num_b) * NUM_NOISE
        x_cat_in = x_cat_b.masked_fill(torch.rand_like(x_cat_b.float()) < CAT_NOISE, 0)
        mu, log_sigma, cat_logits = model_ae(x_num_in, x_cat_in)
        loss = mixed_nll_loss(x_num_b, x_cat_b, mu, log_sigma, cat_logits, CONT_WEIGHT)
        optimizer_ae.zero_grad(); loss.backward(); optimizer_ae.step()
        epoch_loss.append(loss.item())
    train_losses.append(np.mean(epoch_loss))
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{EPOCHS} — loss: {train_losses[-1]:.4f}")


plt.plot(train_losses)
plt.xlabel('Epoch'); plt.ylabel('NLL Loss'); plt.title('Mixed Autoencoder Training Loss')
plt.show()


### Advanced Autoencoder — Model Evaluation

In [ ]:
scores_val          = compute_anomaly_scores(model_ae, X_val_num_t, X_val_cat_t, CONT_WEIGHT)
scores_train_normal = compute_anomaly_scores(model_ae, X_tr_num_t, X_tr_cat_t, CONT_WEIGHT)

for pct in [99.0, 99.5, 99.9]:
    t  = threshold_from_normal_scores(scores_train_normal, pct)
    f1 = f1_score(y_val_np, (scores_val >= t).astype(int), zero_division=0)
    print(f"  {pct:.1f}th pct  thr={t:.3f}  F1={f1:.4f}")

thr = threshold_from_normal_scores(scores_train_normal, percentile=99.5)

evaluate_model(
    y_val_np, scores_val,
    f"Mixed Autoencoder (cw={CONT_WEIGHT})",
    threshold=thr
)
